# Bibliometrix-Python ETL Pipeline — Demonstration Notebook

**Data Science 2025/2026 — Prof. Moscato**  
**Student:** Raed Eleyan  

This notebook demonstrates the full ETL (Extract → Transform → Load) pipeline that makes
the bibliometrix-python Shiny dashboard source-agnostic. It replicates the R package's
`convert2df()` function for five data sources:

| Source | Mode | Format |
|--------|------|--------|
| Web of Science | File | `.txt` tagged format |
| Scopus | File | `.csv` |
| Dimensions | File | `.xlsx` |
| PubMed | File | `.txt` MEDLINE format |
| OpenAlex API | Live API | REST JSON |
| PubMed API | Live API | E-utilities XML |

## 1. Package Structure

```
www/services/etl/
├── __init__.py       Public API
├── schema.py         WoS tag schema & type contracts
├── mappings.py       Source → WoS column mapping dicts
├── extractor.py      File loading & source detection
├── api_retriever.py  OpenAlex & PubMed E-utilities fetcher
├── standardizer.py   Rename / normalize / type-enforce
├── sr_generator.py   Short Reference (SR) field computation
├── validator.py      Schema completeness & type checks
├── exporter.py       CSV serialization
└── pipeline.py       High-level orchestrators
```

## 2. Setup

In [ ]:
import os, sys, warnings
import pandas as pd

# Ensure the project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

In [ ]:
from www.services.etl import (
    run_file_pipeline,
    run_api_pipeline,
    convert2df,
    REQUIRED_COLUMNS,
    MULTI_VALUE_COLUMNS,
)

print(f"ETL package loaded. Required columns: {len(REQUIRED_COLUMNS)}")
print("Required:", REQUIRED_COLUMNS)

## 3. Schema Overview

The unified schema uses Web of Science (WoS) 2–3 letter field tags.
Every output DataFrame, regardless of source, must contain exactly these columns.

In [ ]:
from www.services.etl.schema import COLUMN_DESCRIPTIONS, INT_COLUMNS, SCALAR_COLUMNS

schema_df = pd.DataFrame([
    {
        "Tag": col,
        "Description": COLUMN_DESCRIPTIONS.get(col, ""),
        "Type": "int" if col in INT_COLUMNS else "list[str]" if col in MULTI_VALUE_COLUMNS else "str",
        "Required": col in REQUIRED_COLUMNS,
    }
    for col in REQUIRED_COLUMNS
])
schema_df

## 4. Mapping Dictionaries

Each source has a dedicated mapping dictionary that translates raw column names to WoS tags.

In [ ]:
from www.services.etl.mappings import SOURCE_MAPPINGS

print("Available source mappings:")
for source, mapping in SOURCE_MAPPINGS.items():
    active = {k: v for k, v in mapping.items() if v is not None}
    print(f"  {source:12s}  {len(mapping)} raw cols → {len(active)} WoS tags")

In [ ]:
# Inspect Scopus mapping
from www.services.etl.mappings import SCOPUS_MAP
pd.DataFrame([(k, v) for k, v in SCOPUS_MAP.items() if v is not None],
             columns=["Scopus column", "WoS tag"])

## 5. File Mode — Scopus CSV

In [ ]:
SCOPUS_PATH = "sources/Scopus/Scopus.csv"

if os.path.isfile(SCOPUS_PATH):
    df_scopus, valid_scopus, errors_scopus = run_file_pipeline(SCOPUS_PATH)
    print(f"Shape: {df_scopus.shape}")
    print(f"Valid: {valid_scopus}")
    print(f"DB:    {df_scopus['DB'].iloc[0]}")
    if errors_scopus:
        print("Errors:", errors_scopus[:3])
else:
    print("Scopus sample not found at", SCOPUS_PATH)

In [ ]:
if os.path.isfile(SCOPUS_PATH):
    # Check type contracts
    print("TC dtype:", df_scopus["TC"].dtype)
    print("AU sample:", df_scopus["AU"].iloc[0])
    print("SR sample:", df_scopus["SR"].iloc[0])
    df_scopus[["TI", "PY", "SO", "TC", "AU", "SR"]].head(3)

## 6. File Mode — Dimensions XLSX

In [ ]:
DIMENSIONS_PATH = "sources/Dimensions/Dimensions.xlsx"

if os.path.isfile(DIMENSIONS_PATH):
    df_dim, valid_dim, errors_dim = run_file_pipeline(DIMENSIONS_PATH)
    print(f"Shape: {df_dim.shape}")
    print(f"Valid: {valid_dim}")
    print(f"DB:    {df_dim['DB'].iloc[0]}")
    df_dim[["TI", "PY", "TC", "AU"]].head(3)
else:
    print("Dimensions sample not found at", DIMENSIONS_PATH)

## 7. File Mode — PubMed TXT

In [ ]:
PUBMED_PATH = "sources/PubMed/pubmed-allergicrh-set.txt"

if os.path.isfile(PUBMED_PATH):
    df_pm, valid_pm, errors_pm = run_file_pipeline(PUBMED_PATH)
    print(f"Shape: {df_pm.shape}")
    print(f"Valid: {valid_pm}")
    print(f"DB:    {df_pm['DB'].iloc[0]}")
    df_pm[["TI", "PY", "SO", "AU"]].head(3)
else:
    print("PubMed sample not found at", PUBMED_PATH)

## 8. API Mode — OpenAlex

In [ ]:
# Live API call — requires internet
try:
    df_oa, valid_oa, errors_oa = run_api_pipeline(
        query="bibliometrics scientometrics",
        platform="openalex",
        max_records=10,
    )
    print(f"Shape: {df_oa.shape}")
    print(f"Valid: {valid_oa}")
    print(f"DB:    {df_oa['DB'].iloc[0]}")
    df_oa[["TI", "PY", "TC", "AU"]].head(5)
except Exception as e:
    print(f"API call failed (offline?): {e}")

## 9. API Mode — PubMed API

In [ ]:
try:
    df_pm_api, valid_pm_api, errors_pm_api = run_api_pipeline(
        query="allergic rhinitis treatment",
        platform="pubmed_api",
        max_records=10,
    )
    print(f"Shape: {df_pm_api.shape}")
    print(f"Valid: {valid_pm_api}")
    df_pm_api[["TI", "PY", "SO", "PMID"]].head(5)
except Exception as e:
    print(f"API call failed (offline?): {e}")

## 10. Validation Deep-Dive

In [ ]:
from www.services.etl.validator import validate_dataframe
import pandas as pd

# Build an intentionally broken DataFrame
bad_df = pd.DataFrame([{
    "TI": "Hello World",
    "TC": "not-an-int",    # wrong type
    "AU": "Smith J",       # should be a list
    "PY": "22",            # not 4-digit
}])

is_valid, errors = validate_dataframe(bad_df)
print(f"Valid: {is_valid}")
print(f"\n{len(errors)} error(s):")
for err in errors:
    print(" •", err)

## 11. CSV Export

In [ ]:
SCOPUS_PATH = "sources/Scopus/Scopus.csv"
OUT_PATH = "/tmp/bibliometrix_etl_demo.csv"

if os.path.isfile(SCOPUS_PATH):
    df_export, _, _ = run_file_pipeline(SCOPUS_PATH, output_path=OUT_PATH)
    print(f"Exported {len(df_export)} rows to {OUT_PATH}")

    # Reload and verify
    reloaded = pd.read_csv(OUT_PATH)
    print(f"Reloaded shape: {reloaded.shape}")
    print("AU column (serialized):", reloaded["AU"].iloc[0])
else:
    print("Scopus sample not found")

## 12. SR Field Format

The Short Reference (SR) field acts as a unique key per document, using the format:
`FirstAuthorLastName PY, YEAR, JOURNAL_ABBREV`

In [ ]:
import re

SCOPUS_PATH = "sources/Scopus/Scopus.csv"
if os.path.isfile(SCOPUS_PATH):
    df_check, _, _ = run_file_pipeline(SCOPUS_PATH)
    sr_pattern = re.compile(r".+,\s*\d{4},\s*.+")
    valid_sr = df_check["SR"].apply(lambda x: bool(sr_pattern.match(str(x))))
    print(f"SR format valid: {valid_sr.sum()}/{len(df_check)} records")
    print("\nSample SR values:")
    print(df_check["SR"].head(5).to_string())

## 13. CLI Usage

The `run_etl.py` script provides a command-line interface:

In [ ]:
import subprocess, sys

# File mode example
SCOPUS_PATH = "sources/Scopus/Scopus.csv"
if os.path.isfile(SCOPUS_PATH):
    result = subprocess.run(
        [sys.executable, "run_etl.py", "--mode", "file", "--input", SCOPUS_PATH],
        cwd=os.path.abspath("."),
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode not in (0, 2):
        print("STDERR:", result.stderr[:500])

## 14. Source-Agnostic Schema Comparison

All sources produce a DataFrame with the same required columns.

In [ ]:
sources = {
    "Scopus": "sources/Scopus/Scopus.csv",
    "Dimensions": "sources/Dimensions/Dimensions.xlsx",
    "PubMed": "sources/PubMed/pubmed-allergicrh-set.txt",
}

rows = []
for name, path in sources.items():
    if os.path.isfile(path):
        df, is_valid, errors = run_file_pipeline(path)
        rows.append({
            "Source": name,
            "Records": len(df),
            "Columns": len(df.columns),
            "Required present": sum(1 for c in REQUIRED_COLUMNS if c in df.columns),
            "Required total": len(REQUIRED_COLUMNS),
            "Validation": "PASS" if is_valid else f"WARN ({len(errors)} issues)",
            "DB": df["DB"].iloc[0] if len(df) > 0 else "?",
        })
    else:
        rows.append({"Source": name, "Records": "(file missing)"})

pd.DataFrame(rows)